# FAC_NB_Demo_Libreria_C — módulo .py en Files del Lakehouse + sys.path

**Variante 3.** El archivo `fac_utils_lh.py` está en `Files/libs/` de `FAC_LH_Utils` (Lakehouse predeterminado
de este notebook, montado en `/lakehouse/default`). Se agrega esa carpeta a `sys.path` y se importa.

Límites que conviene mostrar: no hay versionado de Fabric para el archivo (Git del Lakehouse solo registra
metadatos, no el contenido de Files), `sys.path` solo afecta al intérprete del **driver** y cualquier cambio
al archivo requiere `importlib.reload` o reiniciar el intérprete.

In [ ]:
p_variante = "C_files"
p_lakehouse_origen = "AB_Lakehouse"
p_tabla_transacciones = "ab_silver_transacciones"
p_ruta_modulo = "/lakehouse/default/Files/libs"

## Medir la importación

In [ ]:
import os
import sys
import time

sys.dont_write_bytecode = True   # evita que Python deje __pycache__ dentro de Files/libs
if not os.path.isdir(p_ruta_modulo):
    raise FileNotFoundError(
        f"{p_ruta_modulo} no existe: suba fac_utils_lh.py a Files/libs de FAC_LH_Utils y use ese Lakehouse como predeterminado"
    )
if p_ruta_modulo not in sys.path:
    sys.path.insert(0, p_ruta_modulo)

_t0 = time.perf_counter()
import fac_utils_lh
t_import = time.perf_counter() - _t0

tiempos = {"import_fac_utils_lh_s": round(t_import, 3)}
verificaciones = {"ruta_modulo": fac_utils_lh.__file__, "fac_utils_lh_version": fac_utils_lh.__version__}
print(f"import fac_utils_lh: {t_import:.3f} s")
print("fac_utils_lh", fac_utils_lh.__version__, "->", fac_utils_lh.__file__)

In [ ]:
from fac_utils_lh import (
    version,
    normalizar_columnas,
    validar_rango_fechas,
    hash_fila,
    hash_fila_python,
    escribir_bitacora,
    leer_bitacora,
    leer_tabla_delta,
)

Para ver un cambio del archivo en la misma sesión: `import importlib; importlib.reload(fac_utils_lh)`.

## Usar las funciones (este bloque es idéntico en las tres variantes)

Las cuatro utilidades de la demo: **normalizar columnas**, **validar un rango de fechas**,
**hash de fila para conciliar** y **escribir en la bitácora**. Los datos son sintéticos.

In [ ]:
import json

datos = [("TX1", "2026-04-01", 100.50, 1), ("TX2", "2026-09-30", 20.00, 2), ("TX3", "2026-10-05", 5.25, 3)]
df_sucio = spark.createDataFrame(datos, ["ID Transacción", "Fecha de Alta (UTC)", "Monto (CRC)", "Monto  CRC"])
df_limpio = normalizar_columnas(df_sucio)
print("Antes:  ", df_sucio.columns)
print("Después:", df_limpio.columns)   # esperado: id_transaccion, fecha_de_alta_utc, monto_crc, monto_crc_2
verificaciones["columnas_normalizadas"] = df_limpio.columns == ["id_transaccion", "fecha_de_alta_utc", "monto_crc", "monto_crc_2"]

In [ ]:
r_ejemplo = validar_rango_fechas(df_limpio, "fecha_de_alta_utc", "2026-04-01", "2026-09-30")
print(r_ejemplo)   # esperado: total 3, fuera_de_rango 1 (2026-10-05), ok False

tx = leer_tabla_delta(spark, p_lakehouse_origen, p_tabla_transacciones)   # solo lectura
r_tx = validar_rango_fechas(tx, "fecha_hora", "2026-04-01", "2026-09-30")
print(r_tx)        # esperado con el dataset del laboratorio: total 72000, nulos 0, fuera_de_rango 0, ok True
verificaciones["rango_ejemplo_fuera_de_rango"] = r_ejemplo["fuera_de_rango"]
verificaciones["rango_tx"] = {k: r_tx[k] for k in ("total", "nulos", "fuera_de_rango", "ok")}

In [ ]:
columnas_hash = ["transaccion_id", "cuenta_id", "direccion", "canal", "monto"]
tx_h = hash_fila(tx, columnas_hash)

# SHA-256 esperado de las tres primeras transacciones (dataset sintético del laboratorio).
ESPERADO = {
    "TX00000001": "4d01750fa7d9891609de2ddfaca0af9f435d9037944b6d3d49948a8cbea76a50",
    "TX00000002": "967eb9eeb586c00246cccfccdd521ce6f72750fa24275b6850caf7ad9697ff99",
    "TX00000003": "bcec1f1f890d7e1d57e575cdf6fa6a5902f8c7d2061ff70e0e5fef85e0aff819",
}
coincidencias = []
for fila in tx_h.orderBy("transaccion_id").limit(3).collect():
    en_python = hash_fila_python([fila[c] for c in columnas_hash])
    igual_python = fila["hash_fila"] == en_python
    igual_referencia = fila["hash_fila"] == ESPERADO.get(fila["transaccion_id"])
    coincidencias.append(igual_python and igual_referencia)
    print(fila["transaccion_id"], fila["hash_fila"][:16] + "…",
          "| igual a hash_fila_python:", igual_python, "| igual al valor de referencia:", igual_referencia)
hashes_distintos = tx_h.select("hash_fila").distinct().count()
print("hashes distintos:", hashes_distintos, "de", r_tx["total"])
verificaciones["hash_tres_primeras_ok"] = all(coincidencias) and len(coincidencias) == 3
verificaciones["hashes_distintos"] = hashes_distintos

In [ ]:
escribir_bitacora(spark, f"demo_libreria_{p_variante}", "OK", filas=r_tx["total"],
                  mensaje=json.dumps({"tiempos": tiempos, "verificaciones": verificaciones}))
display(leer_bitacora(spark, ultimas=3))

In [ ]:
resumen = {
    "variante": p_variante,
    "version_funciones": version(),
    "validacion_fechas_ok": r_tx["ok"],
    "filas": r_tx["total"],
    "tiempos": tiempos,
    "verificaciones": verificaciones,
}
print(json.dumps(resumen, indent=2, ensure_ascii=False))